Tarea 4: Chunking fijo (application/services/retrieval_service.py)
Implementa la función `overlap_chunking` que recibe texto crudo y parámetros de chunking,
y devuelve una `list[Chunk]` (modelo de dominio). Chunking fijo: 500 caracteres con 50 de overlap.
Maneja el caso borde de texto más corto que chunk_size (retorna 1 solo chunk).
Escribe tests unitarios parametrizados con múltiples escenarios:
texto corto, texto largo, texto de exactamente chunk_size, y texto de chunk_size+1.

Una vez se tienen los chunks se debe modificar hacia el objeto Document que es estandar en el uso de bases vectoriales y que está declarado como el objeto genérico para el protocolo VectorStore

In [67]:
# Use this initial code to work in the notebook as if it were a module, that
# is, to be able to export classes and functions from other subpackages.

import os
import sys

package_path = os.path.abspath(".").split(os.sep + "notebooks")[0]
if package_path not in sys.path:
    sys.path.append(package_path)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [68]:
from researchos.domain.models import Chunk

def overlap_chunking(
    text: str, paper_id: str, 
    chunk_size: int = 500, overlap: int = 50
    ) -> list[Chunk]:

    
    chunks = []
    
    for i, initial_car in enumerate(range(0, len(text), chunk_size)):

        start_chunk = initial_car - overlap*i
        end_chunk = start_chunk + chunk_size
        text_chunk = text[start_chunk:end_chunk]
        chunk_id = f"{paper_id}_{i}"
        start_char = start_chunk
        end_char = min(end_chunk, len(text))

        chunk = Chunk(
            chunk_id=chunk_id,
            paper_id=paper_id,
            text=text_chunk,
            metadata={
                'chunk_size': chunk_size,
                'overlap': overlap,
                'start_char': start_char,
                'end_char': end_char
            },
            chunk_index=i
        )

        chunks.append(chunk)
    return chunks

In [69]:
full_text = "The rapid advancement of artificial intelligence has transformed numerous fields, from healthcare to finance, enabling unprecedented capabilities in data analysis and decision-making. Machine learning algorithms, particularly deep neural networks, have demonstrated remarkable performance in tasks such as image recognition, natural language processing, and strategic game playing. These developments have sparked both excitement and concern among researchers, policymakers, and the general public. The integration of AI systems into critical infrastructure raises important questions about reliability, security, and accountability. Researchers are actively working on methods to make AI systems more interpretable and explainable, addressing the black box problem that has long plagued complex models. Federated learning and differential privacy techniques are being developed to enable AI training on sensitive data while preserving user privacy. Meanwhile, the environmental impact of training large models has prompted investigation into more efficient architectures and training procedures. Reinforcement learning from human feedback has emerged as a promising approach for aligning AI behavior with human values and preferences. The development of foundation models trained on massive datasets has enabled few-shot and zero-shot learning across diverse tasks. As these systems become more capable, the importance of robust evaluation frameworks and safety measures continues to grow. The scientific community is increasingly focused on developing AI that is not only powerful but also trustworthy and beneficial to society."

len(full_text)

1630

In [70]:
from IPython.display import display, Markdown
display(Markdown(full_text))


The rapid advancement of artificial intelligence has transformed numerous fields, from healthcare to finance, enabling unprecedented capabilities in data analysis and decision-making. Machine learning algorithms, particularly deep neural networks, have demonstrated remarkable performance in tasks such as image recognition, natural language processing, and strategic game playing. These developments have sparked both excitement and concern among researchers, policymakers, and the general public. The integration of AI systems into critical infrastructure raises important questions about reliability, security, and accountability. Researchers are actively working on methods to make AI systems more interpretable and explainable, addressing the black box problem that has long plagued complex models. Federated learning and differential privacy techniques are being developed to enable AI training on sensitive data while preserving user privacy. Meanwhile, the environmental impact of training large models has prompted investigation into more efficient architectures and training procedures. Reinforcement learning from human feedback has emerged as a promising approach for aligning AI behavior with human values and preferences. The development of foundation models trained on massive datasets has enabled few-shot and zero-shot learning across diverse tasks. As these systems become more capable, the importance of robust evaluation frameworks and safety measures continues to grow. The scientific community is increasingly focused on developing AI that is not only powerful but also trustworthy and beneficial to society.

In [71]:
import os
import fitz

from researchos.paths import PAPERS_DIR

# entries = os.listdir(PAPERS_DIR)
# local_pdf_path = PAPERS_DIR / entries[0]

# full_text = ""
# doc = fitz.open(local_pdf_path)
# for page in doc:
#     full_text += page.get_text()

chunks = overlap_chunking(text=full_text, paper_id=entries[0])
chunks

[Chunk(chunk_id='kamil_szczepanik_2025.pdf_0', paper_id='kamil_szczepanik_2025.pdf', text='The rapid advancement of artificial intelligence has transformed numerous fields, from healthcare to finance, enabling unprecedented capabilities in data analysis and decision-making. Machine learning algorithms, particularly deep neural networks, have demonstrated remarkable performance in tasks such as image recognition, natural language processing, and strategic game playing. These developments have sparked both excitement and concern among researchers, policymakers, and the general public. T', metadata={'chunk_size': 500, 'overlap': 50, 'start_char': 0, 'end_char': 500}, chunk_index=0),
 Chunk(chunk_id='kamil_szczepanik_2025.pdf_1', paper_id='kamil_szczepanik_2025.pdf', text='searchers, policymakers, and the general public. The integration of AI systems into critical infrastructure raises important questions about reliability, security, and accountability. Researchers are actively working on 

In [72]:
chunks[0].text

'The rapid advancement of artificial intelligence has transformed numerous fields, from healthcare to finance, enabling unprecedented capabilities in data analysis and decision-making. Machine learning algorithms, particularly deep neural networks, have demonstrated remarkable performance in tasks such as image recognition, natural language processing, and strategic game playing. These developments have sparked both excitement and concern among researchers, policymakers, and the general public. T'

In [73]:
chunks[1].text

'searchers, policymakers, and the general public. The integration of AI systems into critical infrastructure raises important questions about reliability, security, and accountability. Researchers are actively working on methods to make AI systems more interpretable and explainable, addressing the black box problem that has long plagued complex models. Federated learning and differential privacy techniques are being developed to enable AI training on sensitive data while preserving user privacy. '

In [74]:
chunks[2].text

' on sensitive data while preserving user privacy. Meanwhile, the environmental impact of training large models has prompted investigation into more efficient architectures and training procedures. Reinforcement learning from human feedback has emerged as a promising approach for aligning AI behavior with human values and preferences. The development of foundation models trained on massive datasets has enabled few-shot and zero-shot learning across diverse tasks. As these systems become more capa'

In [75]:
chunks[3].text

's diverse tasks. As these systems become more capable, the importance of robust evaluation frameworks and safety measures continues to grow. The scientific community is increasingly focused on developing AI that is not only powerful but also trustworthy and beneficial to society.'

# Transformar de chunks a Document

In [76]:
from researchos.domain.models import Document

def chunk_to_document(chunk: Chunk) -> Document:
    return Document(
        doc_id=chunk.chunk_id,
        text=chunk.text,
        metadata={**chunk.metadata, "paper_id": chunk.paper_id, "chunk_index": chunk.chunk_index},
    )
    

In [77]:
[chunk_to_document(c) for c in chunks]

[Document(doc_id='kamil_szczepanik_2025.pdf_0', text='The rapid advancement of artificial intelligence has transformed numerous fields, from healthcare to finance, enabling unprecedented capabilities in data analysis and decision-making. Machine learning algorithms, particularly deep neural networks, have demonstrated remarkable performance in tasks such as image recognition, natural language processing, and strategic game playing. These developments have sparked both excitement and concern among researchers, policymakers, and the general public. T', metadata={'chunk_size': 500, 'overlap': 50, 'start_char': 0, 'end_char': 500, 'paper_id': 'kamil_szczepanik_2025.pdf', 'chunk_index': 0}, score=0.0),
 Document(doc_id='kamil_szczepanik_2025.pdf_1', text='searchers, policymakers, and the general public. The integration of AI systems into critical infrastructure raises important questions about reliability, security, and accountability. Researchers are actively working on methods to make AI 